# AMPR: Adaptive Multimodal Protein Representation

**Thesis:** Dự đoán chức năng protein dựa trên Deep Learning  
**Author:** Nguyen Viet Hung (20224998)  
**Environment:** Google Colab — Python 3.12.13, T4 GPU (15 GB VRAM)

---

## Pipeline tổng quan

```
Phase 1 ── Setup & Clone
Phase 2 ── Download raw data (sequences, GO annotations, PPI)
Phase 3 ── Preprocess: FASTA + GO labels + splits
Phase 4 ── Precompute embeddings (ProteinBERT, ProstT5, BioBERT, Node2Vec)
Phase 5 ── Build GO DAG matrix
Phase 6 ── Create TFRecord files (no contact maps)
Phase 7 ── Train: MF model
Phase 8 ── Train: BP model
Phase 9 ── Train: CC model
Phase 10 ─ Evaluate & Visualize results
```

> **Lưu ý:** Phases 4–6 chỉ cần chạy **một lần**. Lưu kết quả lên Google Drive để tái sử dụng khi session Colab reset.

---
## Phase 0 — Kiểm tra GPU

Đảm bảo runtime đang dùng GPU T4. Nếu không thấy GPU, vào **Runtime → Change runtime type → GPU**.

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'GPU không khả dụng — kiểm tra runtime type!')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Sat Apr 25 02:31:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## Phase 1 — Setup: Clone repo & cài thư viện

Clone repo AMPR và cài đặt các dependencies theo `requirements.txt`.  
Tất cả phiên bản được pin cứng để đảm bảo tái lập kết quả.

In [6]:
import os

REPO_URL = 'https://github.com/hungithust/datn_protein_function.git'  # <-- thay bằng URL repo của bạn
WORK_DIR = '/content/ampr'

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    print(f'Repo đã tồn tại tại {WORK_DIR}')
    !git -C {WORK_DIR} pull

os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

Repo đã tồn tại tại /content/ampr
Already up to date.
Working directory: /content/ampr


In [3]:
# Cài dependencies — dùng Colab pre-installed cho torch/numpy/tensorflow để tránh conflict
# với torchvision, torchaudio, jax, scikit-learn, v.v... mà Colab đã pre-install.
#
# Chỉ install những package KHÔNG có sẵn trong Colab hoặc cần phiên bản cụ thể:
#   - transformers, obonet, biopython, pyyaml, tqdm
#   - dgl (cần wheel riêng cho CUDA)
#
# Torch/numpy/tensorflow/scipy/sklearn/matplotlib dùng phiên bản Colab đã cài sẵn.

import sys, subprocess

# Install core packages không có sẵn, KHÔNG pin torch/numpy
!pip install -q --no-warn-conflicts \
    transformers==4.41.2 \
    obonet==1.0.0 \
    biopython==1.84 \
    pyyaml==6.0.1 \
    tqdm==4.66.4

# DGL: install wheel tương ứng với torch hiện có của Colab
# (Colab 2026 dùng torch 2.10/CUDA 12.8, nhưng DGL 2.1 wheel vẫn chạy)
!pip install -q --no-warn-conflicts dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html

print('\n✓ Thư viện chính đã cài xong (dùng Colab defaults cho torch/numpy/tensorflow)')


✓ Thư viện chính đã cài xong (dùng Colab defaults cho torch/numpy/tensorflow)


In [2]:
# Xác nhận versions — chấp nhận mọi phiên bản Colab provide
import sys
import torch, transformers, numpy, sklearn, scipy

print(f'Python       : {sys.version.split()[0]}')
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'numpy        : {numpy.__version__}')
print(f'scipy        : {scipy.__version__}')
print(f'sklearn      : {sklearn.__version__}')

try:
    import tensorflow as tf
    print(f'tensorflow   : {tf.__version__}')
except Exception as e:
    print(f'tensorflow   : (lazy import) — {type(e).__name__}')

try:
    import dgl
    print(f'dgl          : {dgl.__version__}')
except Exception as e:
    print(f'dgl          : LOAD FAILED — {e}')

# Verify NumPy 2.x compatibility: một số API cũ (np.int_, np.bool_) đã bị xóa
# nhưng code AMPR chỉ dùng np.zeros/np.array/np.float32 (stable APIs)
if numpy.__version__.startswith('2.'):
    print('\n[INFO] NumPy 2.x — sẽ hoạt động với code AMPR (dùng stable APIs only)')

Python       : 3.12.13
torch        : 2.10.0+cu128
transformers : 4.41.2
numpy        : 2.0.2
scipy        : 1.16.3
sklearn      : 1.6.1
tensorflow   : 2.19.0
dgl          : 2.5.0+cu121

[INFO] NumPy 2.x — sẽ hoạt động với code AMPR (dùng stable APIs only)


---
## Phase 2 — Mount Google Drive (tuỳ chọn nhưng khuyến nghị)

Mount Drive để lưu dữ liệu lớn (embeddings ~2-5 GB) giữa các session.  
Nếu không mount, mọi dữ liệu sẽ mất khi Colab reset.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục data trên Drive
DATA_DIR = '/content/drive/MyDrive/ampr_data'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(f'{DATA_DIR}/embeddings', exist_ok=True)
os.makedirs(f'{DATA_DIR}/tfrecords', exist_ok=True)
os.makedirs(f'{DATA_DIR}/dag_matrices', exist_ok=True)
os.makedirs(f'{DATA_DIR}/checkpoints', exist_ok=True)

# Symlink từ WORK_DIR/data → Drive để code không cần biết về Drive path
if not os.path.exists(f'{WORK_DIR}/data'):
    os.symlink(DATA_DIR, f'{WORK_DIR}/data')

print(f'Data directory: {DATA_DIR}')
print('Drive mounted và symlink tạo xong.')

Mounted at /content/drive
Data directory: /content/drive/MyDrive/ampr_data
Drive mounted và symlink tạo xong.


---
## Phase 3 — Download Raw Data

Download các file cần thiết từ nguồn công khai:

| File | Nguồn | Kích thước | Mục đích |
|---|---|---|---|
| `pdb_seqres.txt.gz` | wwPDB FTP | ~50 MB | Protein sequences |
| `pdb_chain_go.tsv.gz` | EBI SIFTS | ~20 MB | GO annotations |
| `bc-95.out` | RCSB | ~5 MB | Sequence clusters 95% |
| `go-basic.obo` | OBO Library | ~2 MB | GO DAG hierarchy |

> **Thời gian ước tính:** ~10-20 phút (phụ thuộc vào kết nối mạng của Colab)

In [6]:
os.makedirs('data/raw', exist_ok=True)

downloads = [
    ('https://files.rcsb.org/pub/pdb/derived_data/pdb_seqres.txt.gz',
     'data/raw/pdb_seqres.txt.gz'),
    ('ftp://ftp.ebi.ac.uk/pub/databases/msd/sifts/flatfiles/tsv/pdb_chain_go.tsv.gz',
     'data/raw/pdb_chain_go.tsv.gz'),
    ('http://web.archive.org/web/20220202030208/https://cdn.rcsb.org/resources/sequence/clusters/bc-95.out',
     'data/raw/bc-95.out'),
    ('http://purl.obolibrary.org/obo/go/go-basic.obo',
     'data/raw/go-basic.obo'),
]

for url, out_path in downloads:
    if os.path.exists(out_path):
        print(f'[SKIP] {out_path} đã tồn tại')
    else:
        print(f'[DOWN] {url}')
        !wget -q "{url}" -O "{out_path}"
        print(f'  → Saved: {out_path}')

print('\n[DONE] Download hoàn tất. Files:')
!ls -lh data/raw/

[DOWN] https://files.rcsb.org/pub/pdb/derived_data/pdb_seqres.txt.gz
  → Saved: data/raw/pdb_seqres.txt.gz
[DOWN] ftp://ftp.ebi.ac.uk/pub/databases/msd/sifts/flatfiles/tsv/pdb_chain_go.tsv.gz
  → Saved: data/raw/pdb_chain_go.tsv.gz
[DOWN] http://web.archive.org/web/20220202030208/https://cdn.rcsb.org/resources/sequence/clusters/bc-95.out
  → Saved: data/raw/bc-95.out
[DOWN] http://purl.obolibrary.org/obo/go/go-basic.obo
  → Saved: data/raw/go-basic.obo

[DONE] Download hoàn tất. Files:
total 169M
-rw------- 1 root root 4.3M Jan 28  2022 bc-95.out
-rw------- 1 root root  31M Apr  2 02:53 go-basic.obo
-rw------- 1 root root  73M Apr 25 02:33 pdb_chain_go.tsv.gz
-rw------- 1 root root  61M Apr 18 05:05 pdb_seqres.txt.gz


---
## Phase 4 — Preprocess: Tạo FASTA + GO Labels + Splits

Dùng script `create_nrPDB_GO_annot.py` từ DeepFRI để:
1. Lọc protein có chiều dài 60–1000 aa
2. Loại bỏ protein trùng lặp (90% sequence identity clustering)
3. Propagate GO terms theo DAG hierarchy (parent terms)
4. Tạo train/valid/test splits (90%/10%/5000 proteins cố định)

Output:
- `nrPDB-GO_annot.tsv` — mapping protein → GO terms (MF/BP/CC)
- `nrPDB-GO_sequences.fasta` — sequences
- `nrPDB-GO_train/valid/test.txt` — split files

In [7]:
# Clone DeepFRI chỉ để lấy preprocessing script
if not os.path.exists('/content/DeepFRI'):
    !git clone -q https://github.com/flatironinstitute/DeepFRI /content/DeepFRI
    print('DeepFRI cloned')
else:
    print('DeepFRI đã có')

DeepFRI cloned


### Patch DeepFRI script cho Biopython >= 1.78

`Bio.Alphabet` bị xóa từ Biopython 1.78 (2020). Script DeepFRI dùng `from Bio.Alphabet import generic_protein`
sẽ raise `ImportError` trên Python 3.12 + Biopython 1.84.

Cell dưới đây patch script trực tiếp trước khi chạy — **không cần sửa file gốc**.

In [4]:
import re

SCRIPT_PATH = '/content/DeepFRI/preprocessing/create_nrPDB_GO_annot.py'

with open(SCRIPT_PATH, 'r') as f:
    src = f.read()

original = src

# Fix 1: Xóa dòng import Bio.Alphabet (không tồn tại trong Biopython >= 1.78)
src = re.sub(r'^from Bio\.Alphabet import.*\n?', '', src, flags=re.MULTILINE)
src = re.sub(r'^import Bio\.Alphabet.*\n?',      '', src, flags=re.MULTILINE)

# Fix 2: Seq(seq, generic_protein)  →  Seq(seq)
src = re.sub(r'Seq\(([^,)]+),\s*generic_protein\)', r'Seq(\1)', src)

# Fix 3: Seq(seq, alphabet=generic_protein)  →  Seq(seq)
src = re.sub(r'Seq\(([^,)]+),\s*alphabet\s*=\s*generic_protein\)', r'Seq(\1)', src)

# Fix 4: KeyError khi chain trong pdb2go nhưng bị lọc khỏi pdb2seq
# (sequence quá ngắn/dài hoặc có ký tự không chuẩn)
# Thay vòng lặp trong write_output_files để skip chain không có trong pdb2seq
#
# Pattern gốc (DeepFRI):
#   sequences_list.append(SeqRecord(Seq(pdb2seq[chain]), id=chain, description="nrPDB"))
# → Thêm guard: chỉ append nếu chain có trong pdb2seq
src = re.sub(
    r'([ \t]*)sequences_list\.append\(SeqRecord\(Seq\(pdb2seq\[chain\]\)',
    r'\1if chain not in pdb2seq:\n\1    continue\n\1sequences_list.append(SeqRecord(Seq(pdb2seq[chain])',
    src
)

if src == original:
    print('[PATCH] Không tìm thấy pattern nào — script có thể đã tương thích.')
else:
    with open(SCRIPT_PATH, 'w') as f:
        f.write(src)
    print('[PATCH] OK — các fix đã được áp dụng:')
    print('  Fix 1+2+3: Bio.Alphabet removed (Biopython >= 1.78)')
    print('  Fix 4    : KeyError pdb2seq[chain] — skip chains bị lọc khỏi sequences')

# Verify: không còn reference tới Bio.Alphabet
remaining_alphabet = [l for l in src.splitlines() if 'Alphabet' in l or 'generic_protein' in l]
if remaining_alphabet:
    print('[WARN] Còn sót Bio.Alphabet — kiểm tra thủ công:')
    for l in remaining_alphabet:
        print(f'  {l}')
else:
    print('[OK] Không còn Bio.Alphabet reference.')

# Verify Fix 4: kiểm tra guard đã có trong source
if 'if chain not in pdb2seq' in src:
    print('[OK] KeyError guard đã được patch.')
else:
    print('[WARN] Fix 4 không thành công — kiểm tra pattern trong script gốc:')
    # In context xung quanh dòng sequences_list để debug
    for i, line in enumerate(src.splitlines()):
        if 'sequences_list' in line and 'SeqRecord' in line:
            print(f'  Line {i}: {line}')

print('\n[READY] Script sẵn sàng chạy preprocessing.')

[PATCH] OK — các fix đã được áp dụng:
  Fix 1+2+3: Bio.Alphabet removed (Biopython >= 1.78)
  Fix 4    : KeyError pdb2seq[chain] — skip chains bị lọc khỏi sequences
[OK] Không còn Bio.Alphabet reference.
[OK] KeyError guard đã được patch.

[READY] Script sẵn sàng chạy preprocessing.


In [9]:
# Cài dependency của DeepFRI preprocessing
!pip install -q obonet networkx

ANNOT_PREFIX = 'data/raw/nrPDB-GO'

# Check đủ cả 4 output files — nếu thiếu bất kỳ file nào thì chạy lại
required_outputs = [
    f'{ANNOT_PREFIX}_annot.tsv',
    f'{ANNOT_PREFIX}_train.txt',
    f'{ANNOT_PREFIX}_valid.txt',
    f'{ANNOT_PREFIX}_test.txt',
]
missing = [f for f in required_outputs if not os.path.exists(f)]

if not missing:
    print('[SKIP] Tất cả preprocessing outputs đã tồn tại')
else:
    # Xóa file partial nếu có (script crash giữa chừng → chỉ có annot.tsv)
    for f in required_outputs:
        if os.path.exists(f):
            os.remove(f)
            print(f'[RM] Xóa file partial: {f}')

    print('[RUN] create_nrPDB_GO_annot.py — mất ~10-15 phút...')
    !python /content/DeepFRI/preprocessing/create_nrPDB_GO_annot.py \
        -sifts data/raw/pdb_chain_go.tsv.gz \
        -bc    data/raw/bc-95.out \
        -seqres data/raw/pdb_seqres.txt.gz \
        -obo   data/raw/go-basic.obo \
        -out   {ANNOT_PREFIX}
    print('[DONE] Preprocessing xong!')

# Xem kết quả
!ls -lh data/raw/nrPDB-GO*
print('\n--- Train/Valid/Test counts ---')
for split in ['train', 'valid', 'test']:
    f = f'data/raw/nrPDB-GO_{split}.txt'
    if os.path.exists(f):
        n = sum(1 for _ in open(f))
        print(f'  {split:5s}: {n:,} proteins')
    else:
        print(f'  {split:5s}: MISSING')

[RUN] create_nrPDB_GO_annot.py — mất ~10-15 phút...
### nrPDB annotated chains= 50182
### Loading SIFTS annotations...
### molecular_function : 611
### biological_process : 1810
### cellular_component : 371
Total number of annot nrPDB=44422
Total number of test nrPDB=3684
[DONE] Preprocessing xong!
-rw------- 1 root root 9.7M Apr 25 02:35 data/raw/nrPDB-GO_annot.tsv
-rw------- 1 root root  14M Apr 25 02:35 data/raw/nrPDB-GO_sequences.fasta
-rw------- 1 root root 1.5M Apr 25 02:35 data/raw/nrPDB-GO_test_sequences.fasta
-rw------- 1 root root  26K Apr 25 02:35 data/raw/nrPDB-GO_test.txt
-rw------- 1 root root 253K Apr 25 02:35 data/raw/nrPDB-GO_train.txt
-rw------- 1 root root  29K Apr 25 02:35 data/raw/nrPDB-GO_valid.txt

--- Train/Valid/Test counts ---
  train: 36,664 proteins
  valid: 4,074 proteins
  test : 3,684 proteins


In [10]:
# Kiểm tra format của annot.tsv
with open('data/raw/nrPDB-GO_annot.tsv') as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i > 12: break  # chỉ in 13 dòng đầu (headers + 1 data row)

### GO-terms (molecular_function)
GO:0046961	GO:0008106	GO:0016855	GO:0004659	GO:0140318	GO:0030295	GO:0098531	GO:0140313	GO:0010181	GO:0031072	GO:0050897	GO:0017069	GO:0016491	GO:0003916	GO:0016709	GO:0008173	GO:0005164	GO:0005319	GO:0004029	GO:0016712	GO:0170055	GO:0005543	GO:0140171	GO:0004457	GO:0004672	GO:0035639	GO:0008170	GO:0003955	GO:0019899	GO:0015929	GO:0016829	GO:0015925	GO:0015252	GO:0004550	GO:0051539	GO:0008483	GO:0016755	GO:0016616	GO:0008320	GO:0008509	GO:0061783	GO:0008324	GO:0051020	GO:0046982	GO:0016661	GO:0016278	GO:0016836	GO:0005516	GO:0015035	GO:0005217	GO:0016903	GO:0004364	GO:0004521	GO:0004190	GO:0031420	GO:0008186	GO:0140098	GO:0008194	GO:0016410	GO:0003713	GO:0015288	GO:0019871	GO:0070006	GO:0003700	GO:0061629	GO:0016628	GO:0004520	GO:0019207	GO:0000978	GO:0016831	GO:0141047	GO:0004713	GO:0016863	GO:0005524	GO:0015085	GO:0005549	GO:0016849	GO:0097367	GO:0004316	GO:0016874	GO:0016765	GO:0031369	GO:0015038	GO:0016868	GO:0044389	GO:0005262	GO:0004623	GO:004687

---
## Phase 5 — Precompute Embeddings

Chạy một lần, kết quả lưu vào Drive để tái sử dụng.

| Model | Input | Output | Thời gian |
|---|---|---|---|
| **ProteinBERT** | FASTA sequences | `seq_embeddings.npy` (N, 1024) | ~1-2h |
| **ProstT5** | FASTA sequences | `struct_embeddings.npy` (N, 1024) | ~1-2h |
| **BioBERT** | GO term definitions | `go_emb_{mf,bp,cc}.npy` (C, 768) | ~10 min |

> **Quan trọng:** Tất cả proteins phải được lưu theo cùng một thứ tự (protein_order.json).  
> Thứ tự này được dùng để map split protein IDs → hàng trong .npy files.

In [1]:
%cd /content/ampr

/content/ampr


In [2]:
# Bước 5.1: Load sequences từ FASTA và tạo protein_order.json
import json
from Bio import SeqIO

FASTA_PATH = 'data/raw/nrPDB-GO_sequences.fasta'

records = list(SeqIO.parse(FASTA_PATH, 'fasta'))
protein_order = [r.id for r in records]
sequences     = [str(r.seq) for r in records]

print(f'[FASTA] Loaded {len(records)} sequences')
print(f'  Min length : {min(len(s) for s in sequences)}')
print(f'  Max length : {max(len(s) for s in sequences)}')
print(f'  Example ID : {protein_order[0]}')

# Lưu protein order — QUAN TRỌNG để map split IDs → row indices
with open('data/protein_order.json', 'w') as f:
    json.dump(protein_order, f)
print(f'[SAVED] data/protein_order.json ({len(protein_order)} entries)')

[FASTA] Loaded 44422 sequences
  Min length : 60
  Max length : 1000
  Example ID : 6EMI-A
[SAVED] data/protein_order.json (44422 entries)


In [12]:
# Bước 5.2: ProteinBERT sequence embeddings
# Output: data/embeddings/seq_embeddings.npy  shape=(N, 1024)

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

SEQ_EMB_PATH = 'data/embeddings/seq_embeddings.npy'

if os.path.exists(SEQ_EMB_PATH):
    print(f'[SKIP] {SEQ_EMB_PATH} đã tồn tại. shape={np.load(SEQ_EMB_PATH).shape}')
else:
    print('[LOAD] Loading ProteinBERT model...')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained('Rostlab/prot_bert', do_lower_case=False)
    model = AutoModel.from_pretrained('Rostlab/prot_bert').to(device)
    model.eval()

    BATCH_SIZE = 16  # T4 15GB: giảm nếu OOM
    all_embeddings = []

    print(f'[RUN] Encoding {len(sequences)} sequences (batch={BATCH_SIZE})...')
    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc='ProteinBERT'):
            batch_seqs = sequences[i:i+BATCH_SIZE]
            # ProteinBERT cần space giữa các amino acid
            batch_seqs_spaced = [' '.join(s) for s in batch_seqs]

            encoded = tokenizer(
                batch_seqs_spaced,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=1024,
            ).to(device)

            output = model(**encoded)
            # Mean pooling qua sequence length (bỏ [CLS] và [SEP])
            hidden = output.last_hidden_state[:, 1:-1, :]  # (B, L, 1024)
            mask = encoded['attention_mask'][:, 1:-1].unsqueeze(-1).float()
            emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)  # (B, 1024)

            all_embeddings.append(emb.cpu().numpy())

    seq_emb = np.concatenate(all_embeddings, axis=0).astype(np.float32)
    print(f'[OUT]  seq_embeddings shape: {seq_emb.shape}')
    np.save(SEQ_EMB_PATH, seq_emb)
    print(f'[SAVED] {SEQ_EMB_PATH} ({seq_emb.nbytes/1e6:.1f} MB)')

    del model  # Giải phóng VRAM
    torch.cuda.empty_cache()

[LOAD] Loading ProteinBERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

[RUN] Encoding 44422 sequences (batch=16)...


ProteinBERT: 100%|██████████| 2777/2777 [2:07:37<00:00,  2.76s/it]


[OUT]  seq_embeddings shape: (44422, 1024)
[SAVED] data/embeddings/seq_embeddings.npy (182.0 MB)


In [3]:
import os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# Bước 5.3: ProstT5 structural embeddings (từ sequence)
# Output: data/embeddings/struct_embeddings.npy  shape=(N, 1024)

STRUCT_EMB_PATH = 'data/embeddings/struct_embeddings.npy'

if os.path.exists(STRUCT_EMB_PATH):
    print(f'[SKIP] {STRUCT_EMB_PATH} đã tồn tại. shape={np.load(STRUCT_EMB_PATH).shape}')
else:
    print('[LOAD] Loading ProstT5 model...')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained('Rostlab/ProstT5', do_lower_case=False, use_fast=False)
    model = AutoModel.from_pretrained('Rostlab/ProstT5').to(device)
    model.eval()

    BATCH_SIZE = 2   # ProstT5 lớn hơn, batch nhỏ hơn
    all_embeddings = []

    print(f'[RUN] Encoding {len(sequences)} sequences (batch={BATCH_SIZE})...')
    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc='ProstT5'):
            batch_seqs = sequences[i:i+BATCH_SIZE]
            # ProstT5: prefix '<AA2fold>' để encode từ amino acid sequence
            batch_seqs_spaced = ['<AA2fold> ' + ' '.join(s) for s in batch_seqs]

            encoded = tokenizer(
                batch_seqs_spaced,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=512,
            ).to(device)

            output = model(input_ids=encoded['input_ids'],
                           attention_mask=encoded['attention_mask'],
                           decoder_input_ids=encoded['input_ids'])
            # Dùng encoder hidden states
            hidden = output.encoder_last_hidden_state  # (B, L, 1024)
            mask = encoded['attention_mask'].unsqueeze(-1).float()
            emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)  # (B, 1024)

            all_embeddings.append(emb.cpu().numpy())

    struct_emb = np.concatenate(all_embeddings, axis=0).astype(np.float32)
    print(f'[OUT]  struct_embeddings shape: {struct_emb.shape}')
    np.save(STRUCT_EMB_PATH, struct_emb)
    print(f'[SAVED] {STRUCT_EMB_PATH} ({struct_emb.nbytes/1e6:.1f} MB)')

    del model
    torch.cuda.empty_cache()

[LOAD] Loading ProstT5 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have 

[RUN] Encoding 44422 sequences (batch=2)...


ProstT5:   0%|          | 19/22211 [00:27<8:51:51,  1.44s/it]


KeyboardInterrupt: 

In [ ]:
# Bước 5.4: PPI embeddings từ DeepGO graph_new_embeddings.pkl (256-dim, UniProt IDs)
# → map PDB chain IDs sang UniProt qua pdb_chain_go.tsv.gz (đã download)
# Output: data/embeddings/ppi_embeddings.npy  shape=(N, 256)

import pickle, gc
import pandas as pd

PPI_EMB_PATH = 'data/embeddings/ppi_embeddings.npy'
os.makedirs('data/embeddings', exist_ok=True)

if os.path.exists(PPI_EMB_PATH):
    _chk = np.load(PPI_EMB_PATH)
    print(f'[SKIP] {PPI_EMB_PATH} đã tồn tại. shape={_chk.shape}')
else:
    EMB_DIM = 256

    # ── Bước A: Download graph_new_embeddings.pkl → Drive (~7.4 GB) ────────────
    # Lưu vào Drive để tránh phải download lại khi session reset.
    PKL_PATH = '/content/drive/MyDrive/ampr_data/graph_new_embeddings.pkl'
    if not os.path.exists(PKL_PATH):
        print('[DOWN] Downloading graph_new_embeddings.pkl (~7.4 GB) → Drive...')
        print('       Ước tính: 30–60 phút tuỳ tốc độ mạng Colab')
        !wget -q --show-progress \
            http://deepgo.bio2vec.net/data/deepgo/graph_new_embeddings.pkl \
            -O {PKL_PATH}
        print(f'[DONE] Saved: {PKL_PATH}')
    else:
        print(f'[SKIP] {PKL_PATH} đã có trên Drive')

    # ── Bước B: PDB chain → UniProt mapping từ pdb_chain_go.tsv.gz ────────────
    # SIFTS file columns: PDB, CHAIN, SP_PRIMARY (UniProt accession), ...
    SIFTS_PATH = 'data/raw/pdb_chain_go.tsv.gz'
    print(f'\n[MAP] Building PDB chain → UniProt mapping từ {SIFTS_PATH}...')
    sifts = pd.read_csv(
        SIFTS_PATH, sep='\t', comment='#',
        usecols=['PDB', 'CHAIN', 'SP_PRIMARY'],
        dtype=str, low_memory=False
    )
    chain2uni = {}
    for _, row in sifts.iterrows():
        pdb  = str(row['PDB']).upper().strip()
        ch   = str(row['CHAIN']).strip()
        uni  = str(row['SP_PRIMARY']).strip()
        if pdb and ch and uni not in ('nan', 'None', ''):
            key = f'{pdb}-{ch}'
            if key not in chain2uni:
                chain2uni[key] = uni
    del sifts; gc.collect()
    print(f'[MAP] {len(chain2uni):,} PDB chain → UniProt mappings built')

    # Preview coverage on first 500
    covered_preview = sum(1 for p in protein_order[:500] if p in chain2uni)
    print(f'[MAP] Preview coverage (first 500): {covered_preview}/500')

    # ── Bước C: Load pkl → dict UniProt → embedding vector ────────────────────
    print(f'\n[LOAD] Loading {PKL_PATH} ...')
    with open(PKL_PATH, 'rb') as f:
        ppi_raw = pickle.load(f)

    # Normalize: DataFrame (accessions, embeddings) hoặc dict
    if isinstance(ppi_raw, dict):
        uni2emb = ppi_raw
    else:
        uni2emb = dict(zip(ppi_raw['accessions'], ppi_raw['embeddings']))
    del ppi_raw; gc.collect()

    print(f'[PPI]  {len(uni2emb):,} UniProt IDs có trong pkl file')

    # Kiểm tra dim thực tế
    sample_vec = next(iter(uni2emb.values()))
    actual_dim = int(np.asarray(sample_vec).shape[0])
    print(f'[PPI]  Embedding dim = {actual_dim}')
    EMB_DIM = actual_dim

    # ── Bước D: Map protein_order → embeddings ────────────────────────────────
    ppi_emb = np.zeros((len(protein_order), EMB_DIM), dtype=np.float32)
    found = 0
    for i, pid in enumerate(protein_order):
        uni = chain2uni.get(pid)
        if uni and uni in uni2emb:
            vec = np.asarray(uni2emb[uni], dtype=np.float32)
            n = min(len(vec), EMB_DIM)
            ppi_emb[i, :n] = vec[:n]
            found += 1

    coverage = found / len(protein_order) * 100
    print(f'\n[PPI]  Coverage : {found:,}/{len(protein_order):,} ({coverage:.1f}%)')
    print(f'       Zero fallback: {len(protein_order)-found:,} proteins')
    print(f'       Output shape : {ppi_emb.shape}')

    np.save(PPI_EMB_PATH, ppi_emb)
    print(f'[SAVED] {PPI_EMB_PATH} ({ppi_emb.nbytes/1e6:.1f} MB)')

    del uni2emb; gc.collect()
    print(f'\n[NOTE] ppi_dim={EMB_DIM} → configs/{{mf,bp,cc}}.yaml đã được cập nhật ppi_dim: 256')

In [7]:
# Bước 5.5: BioBERT GO term embeddings
# Encode definition text của mỗi GO term → fixed weight matrix cho zero-shot classifier
# Output: data/embeddings/go_emb_{mf,bp,cc}.npy  shape=(C, 768)

import sys
sys.path.insert(0, WORK_DIR)
from scripts.seq2tfrecord import load_GO_annot

import obonet

print('[LOAD] Loading GO annotations and OBO ontology...')
prot2annot, goterms, gonames = load_GO_annot('data/raw/nrPDB-GO_annot.tsv')
go_graph = obonet.read_obo('data/raw/go-basic.obo')

for ont in ['mf', 'bp', 'cc']:
    print(f'\n  {ont.upper()}: {len(goterms[ont])} GO terms')

print('\n[LOAD] Loading BioBERT...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
bio_tokenizer = AutoTokenizer.from_pretrained('dmis-lab/biobert-base-cased-v1.2')
bio_model = AutoModel.from_pretrained('dmis-lab/biobert-base-cased-v1.2').to(device)
bio_model.eval()

def encode_go_terms(term_ids, go_graph, tokenizer, model, batch_size=64):
    """Encode GO term definitions via BioBERT [CLS] token."""
    texts = []
    for tid in term_ids:
        if tid in go_graph.nodes:
            node = go_graph.nodes[tid]
            name = node.get('name', tid)
            defn = node.get('def', '').replace('"', '').split('[')[0].strip()
            texts.append(f'{name}. {defn}' if defn else name)
        else:
            texts.append(tid)

    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc='BioBERT GO'):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=128).to(device)
            out = model(**enc)
            cls = out.last_hidden_state[:, 0, :]  # [CLS] token (B, 768)
            embeddings.append(cls.cpu().numpy())

    return np.concatenate(embeddings, axis=0).astype(np.float32)

for ont in ['mf', 'bp', 'cc']:
    out_path = f'data/embeddings/go_emb_{ont}.npy'
    if os.path.exists(out_path):
        print(f'[SKIP] {out_path} đã tồn tại')
        continue
    print(f'[RUN]  Encoding {len(goterms[ont])} {ont.upper()} GO terms...')
    emb = encode_go_terms(goterms[ont], go_graph, bio_tokenizer, bio_model)
    print(f'[OUT]  go_emb_{ont} shape: {emb.shape}')
    np.save(out_path, emb)
    print(f'[SAVED] {out_path} ({emb.nbytes/1e6:.1f} MB)')

del bio_model
torch.cuda.empty_cache()
print('\n[DONE] Tất cả GO embeddings đã được tạo!')

[LOAD] Loading GO annotations and OBO ontology...

  MF: 611 GO terms

  BP: 1810 GO terms

  CC: 371 GO terms

[LOAD] Loading BioBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

[OUT]  go_emb_mf shape: (611, 768)
[SAVED] data/embeddings/go_emb_mf.npy (1.9 MB)
[RUN]  Encoding 1810 BP GO terms...


BioBERT GO: 100%|██████████| 29/29 [00:11<00:00,  2.60it/s]


[OUT]  go_emb_bp shape: (1810, 768)
[SAVED] data/embeddings/go_emb_bp.npy (5.6 MB)
[RUN]  Encoding 371 CC GO terms...


BioBERT GO: 100%|██████████| 6/6 [00:02<00:00,  2.48it/s]

[OUT]  go_emb_cc shape: (371, 768)
[SAVED] data/embeddings/go_emb_cc.npy (1.1 MB)

[DONE] Tất cả GO embeddings đã được tạo!


---
## Phase 6 — Build GO DAG Matrix + Labels

Tạo:
1. **DAG adjacency matrix** `A[i,j]=1` nếu GO term `j` là parent của term `i` — dùng cho DAG loss
2. **Binary label matrices** `labels_{mf,bp,cc}.npy` — mỗi row là 1 protein, mỗi cột là 1 GO term
3. **splits.json** — mapping split name → danh sách protein IDs

In [ ]:
# Bước 6.1: Build DAG adjacency matrices
import networkx as nx

os.makedirs('data/dag_matrices', exist_ok=True)

for ont in ['mf', 'bp', 'cc']:
    out_path = f'data/dag_matrices/dag_matrix_{ont}.npy'
    if os.path.exists(out_path):
        print(f'[SKIP] {out_path} đã tồn tại')
        continue

    terms = goterms[ont]
    C = len(terms)
    term2idx = {t: i for i, t in enumerate(terms)}

    dag = np.zeros((C, C), dtype=np.float32)
    edges_added = 0

    for term_id in terms:
        if term_id not in go_graph:
            continue
        child_idx = term2idx[term_id]
        # go_graph là directed: edge từ child → parent
        for parent_id in go_graph.successors(term_id):
            if parent_id in term2idx:
                parent_idx = term2idx[parent_id]
                dag[child_idx, parent_idx] = 1.0
                edges_added += 1

    np.save(out_path, dag)
    print(f'[SAVED] {out_path} — shape={dag.shape}, edges={edges_added}')

print('\n[DONE] DAG matrices xong!')

In [ ]:
# Bước 6.2: Build binary label matrices
prot2idx_global = {pid: i for i, pid in enumerate(protein_order)}
N = len(protein_order)

for ont in ['mf', 'bp', 'cc']:
    out_path = f'data/labels_{ont}.npy'
    if os.path.exists(out_path):
        print(f'[SKIP] {out_path} đã tồn tại')
        continue

    terms = goterms[ont]
    C = len(terms)
    labels = np.zeros((N, C), dtype=np.float32)

    annotated = 0
    for pid, annot in prot2annot.items():
        if pid not in prot2idx_global:
            continue
        row = prot2idx_global[pid]
        labels[row] = annot[ont]
        if annot[ont].sum() > 0:
            annotated += 1

    np.save(out_path, labels)
    pos_rate = labels.sum() / (N * C) * 100
    print(f'[SAVED] labels_{ont}.npy — shape={labels.shape}, '
          f'annotated={annotated}/{N}, positive_rate={pos_rate:.2f}%')

In [ ]:
# Bước 6.3: Tạo splits.json
train_ids = [l.strip() for l in open('data/raw/nrPDB-GO_train.txt')]
valid_ids = [l.strip() for l in open('data/raw/nrPDB-GO_valid.txt')]
test_ids  = [l.strip() for l in open('data/raw/nrPDB-GO_test.txt')]

# Chỉ giữ những ID có trong protein_order
prot_set = set(protein_order)
splits = {
    'train': [p for p in train_ids if p in prot_set],
    'valid': [p for p in valid_ids if p in prot_set],
    'test':  [p for p in test_ids  if p in prot_set],
}

with open('data/splits.json', 'w') as f:
    json.dump(splits, f)

print('[SPLITS]')
for split, ids in splits.items():
    print(f'  {split:5s}: {len(ids):,} proteins')

---
## Phase 6.4 — Stratified Test Splits (DeepFRI fair-comparison protocol)

DeepFRI báo cáo Fmax/AUPR ở **5 ngưỡng identity với training set: 30%, 40%, 50%, 70%, 95%**. Để so sánh công bằng, ta tách `test` thành:
- `test_LT_30` — protein test có max identity vs ANY training protein **< 30%** (hardest)
- `test_LT_40` — < 40% (nested: LT_30 ⊂ LT_40)
- `test_LT_50`, `test_LT_70` — tương tự
- `test_LT_95` — < 95% theo cluster RCSB bc-95.out (giống DeepFRI cho ngưỡng 95%)

**Method:**
- LT_95: dùng `bc-95.out` (đã download Phase 3) — test protein vào LT_95 nếu cluster 95% của nó không chứa training protein nào
- LT_30/40/50/70: chạy **MMseqs2** `easy-search` test.fasta vs train.fasta → max identity của mỗi test protein

Output: `data/splits_stratified.json` với keys `train, valid, test, test_LT_30, test_LT_40, test_LT_50, test_LT_70, test_LT_95`.

In [ ]:
# ── Phase 6.4 — Stratified test splits ─────────────────────────────────────
import os, json, gzip, gc
import numpy as np
from collections import defaultdict
from Bio import SeqIO

OUT_PATH = 'data/splits_stratified.json'

if os.path.exists(OUT_PATH):
    print(f'[SKIP] {OUT_PATH} đã tồn tại')
    splits_strat = json.load(open(OUT_PATH))
    for k, v in splits_strat.items():
        print(f'  {k:12s}: {len(v):,} proteins')
else:
    # Reload base splits
    splits = json.load(open('data/splits.json'))
    train_set = set(splits['train'])
    valid_set = set(splits['valid'])
    test_ids  = list(splits['test'])
    test_set  = set(test_ids)
    print(f'[INPUT] train={len(train_set):,}  valid={len(valid_set):,}  test={len(test_ids):,}')

    # ── Step A: LT_95 from bc-95.out ───────────────────────────────────────
    print('\n[LT_95] Parsing bc-95.out (PDB-chain clusters at 95%)...')
    BC95_PATH = 'data/raw/bc-95.out'
    test2cluster_has_train = {}
    cluster_id = 0
    with open(BC95_PATH) as f:
        for line in f:
            members = line.split()  # tokens like "1ABC_A"
            normalized = []
            for m in members:
                # bc-95.out separates with '_'; protein_order uses '-'
                if '_' in m:
                    pdb, chain = m.split('_', 1)
                    normalized.append(f'{pdb.upper()}-{chain}')
            # Does this cluster contain any training protein?
            has_train = any(pid in train_set for pid in normalized)
            for pid in normalized:
                if pid in test_set:
                    # If no train in cluster → goes into LT_95 (no homolog ≥95%)
                    test2cluster_has_train[pid] = has_train or test2cluster_has_train.get(pid, False)
            cluster_id += 1

    # Test proteins not in any bc-95 cluster default to "no homolog" → LT_95
    LT_95 = [p for p in test_ids
             if not test2cluster_has_train.get(p, False)]
    print(f'[LT_95] {len(LT_95):,}/{len(test_ids):,} test proteins have NO ≥95% training homolog')

    # ── Step B: Install MMseqs2 ────────────────────────────────────────────
    print('\n[MMseqs2] Installing...')
    if not os.path.exists('/usr/local/bin/mmseqs'):
        # Static binary — fastest install on Colab
        !wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz -O /tmp/mmseqs.tar.gz
        !tar -xzf /tmp/mmseqs.tar.gz -C /tmp
        !cp /tmp/mmseqs/bin/mmseqs /usr/local/bin/
        !chmod +x /usr/local/bin/mmseqs
    !mmseqs version

    # ── Step C: Build per-split FASTA files ────────────────────────────────
    print('\n[MMseqs2] Writing train.fasta & test.fasta from full FASTA...')
    os.makedirs('data/mmseqs', exist_ok=True)
    train_path = 'data/mmseqs/train.fasta'
    test_path  = 'data/mmseqs/test.fasta'

    if not (os.path.exists(train_path) and os.path.exists(test_path)):
        with open('data/raw/nrPDB-GO_sequences.fasta') as fin, \
             open(train_path, 'w') as ftr, \
             open(test_path, 'w') as fts:
            for rec in SeqIO.parse(fin, 'fasta'):
                if rec.id in train_set:
                    ftr.write(f'>{rec.id}\n{str(rec.seq)}\n')
                elif rec.id in test_set:
                    fts.write(f'>{rec.id}\n{str(rec.seq)}\n')
    print(f'[MMseqs2] train.fasta + test.fasta ready')

    # ── Step D: Run easy-search test vs train ──────────────────────────────
    M8_PATH = 'data/mmseqs/test_vs_train.m8'
    if not os.path.exists(M8_PATH):
        print('[MMseqs2] Running easy-search (test vs train) — ~10-20 min...')
        !mmseqs easy-search {test_path} {train_path} {M8_PATH} data/mmseqs/tmp \
            --min-seq-id 0.0 -c 0.5 --cov-mode 0 --max-seqs 50 \
            --format-output "query,target,pident" -s 7.5 \
            > data/mmseqs/log.txt 2>&1
    print(f'[MMseqs2] Output → {M8_PATH}')

    # ── Step E: Aggregate max-identity per test protein ────────────────────
    print('\n[AGGR] Computing max identity per test protein...')
    max_pident = {}  # test_id → highest identity vs any train
    with open(M8_PATH) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            q, t, pid = parts[0], parts[1], float(parts[2])
            # MMseqs reports identity in [0,1] or [0,100] depending on version — normalize to [0,100]
            if pid <= 1.0:
                pid *= 100.0
            if pid > max_pident.get(q, -1):
                max_pident[q] = pid

    # Test proteins with no MMseqs hit → identity = 0 (no homolog at all)
    for pid_str in test_ids:
        max_pident.setdefault(pid_str, 0.0)

    # ── Step F: Bucket into nested LT_30/40/50/70 ──────────────────────────
    LT_30 = [p for p in test_ids if max_pident[p] < 30.0]
    LT_40 = [p for p in test_ids if max_pident[p] < 40.0]
    LT_50 = [p for p in test_ids if max_pident[p] < 50.0]
    LT_70 = [p for p in test_ids if max_pident[p] < 70.0]

    splits_strat = {
        'train': splits['train'],
        'valid': splits['valid'],
        'test':  splits['test'],
        'test_LT_30': LT_30,
        'test_LT_40': LT_40,
        'test_LT_50': LT_50,
        'test_LT_70': LT_70,
        'test_LT_95': LT_95,
    }
    with open(OUT_PATH, 'w') as f:
        json.dump(splits_strat, f)
    print(f'\n[SAVED] {OUT_PATH}')

    # Summary table
    print('\n[STRATIFIED DISTRIBUTION]')
    n_test = len(test_ids)
    for key in ['test', 'test_LT_30', 'test_LT_40', 'test_LT_50', 'test_LT_70', 'test_LT_95']:
        n = len(splits_strat[key])
        pct = n / n_test * 100
        print(f'  {key:12s}: {n:5d}/{n_test} ({pct:.1f}%)')

    # Quick sanity check: nested ⊆
    for a, b in [('test_LT_30', 'test_LT_40'),
                 ('test_LT_40', 'test_LT_50'),
                 ('test_LT_50', 'test_LT_70')]:
        if not set(splits_strat[a]).issubset(set(splits_strat[b])):
            print(f'[WARN] {a} not subset of {b} — unexpected')

    gc.collect()

---
## Phase 7 — Tạo TFRecord files (không có contact map)

Script `seq2tfrecord.py` chuyển đổi FASTA + GO annotations → TFRecord.  
TFRecord được dùng để kiểm tra pipeline và có thể share với các framework khác (TF, JAX).  

> **Lưu ý:** PyTorch training trong Phase 8-10 sẽ đọc trực tiếp từ `.npy` files đã tạo ở Phase 5-6,  
> không phải từ TFRecord. TFRecord hữu ích nếu bạn muốn compare với TensorFlow baseline.

In [ ]:
os.makedirs('data/tfrecords', exist_ok=True)

for split_name in ['train', 'valid', 'test']:
    prefix = f'data/tfrecords/GO_{split_name}'
    first_shard = f'{prefix}_00-of-10.tfrecords'

    if os.path.exists(first_shard):
        print(f'[SKIP] TFRecords cho {split_name} đã tồn tại')
        continue

    print(f'[RUN] Creating TFRecords for {split_name}...')
    !python scripts/seq2tfrecord.py \
        --annot  data/raw/nrPDB-GO_annot.tsv \
        --fasta  data/raw/nrPDB-GO_sequences.fasta \
        --split  data/raw/nrPDB-GO_{split_name}.txt \
        --out_prefix {prefix} \
        --num_shards 10

print('\n[DONE] TFRecords xong:')
!ls -lh data/tfrecords/ | head -20

In [ ]:
# Verify TFRecord: đọc 1 record để kiểm tra
import tensorflow as tf

# Load annotation để biết số GO terms
_, goterms_chk, _ = load_GO_annot('data/raw/nrPDB-GO_annot.tsv')
n_mf = len(goterms_chk['mf'])
n_bp = len(goterms_chk['bp'])
n_cc = len(goterms_chk['cc'])

first_shard = 'data/tfrecords/GO_train_00-of-10.tfrecords'
raw_dataset = tf.data.TFRecordDataset(first_shard)

feature_desc = {
    'prot_id':   tf.io.FixedLenFeature([], tf.string),
    'L':         tf.io.FixedLenFeature([1], tf.int64),
    'seq_1hot':  tf.io.VarLenFeature(tf.float32),
    'mf_labels': tf.io.FixedLenFeature([n_mf], tf.int64),
    'bp_labels': tf.io.FixedLenFeature([n_bp], tf.int64),
    'cc_labels': tf.io.FixedLenFeature([n_cc], tf.int64),
}

for raw_record in raw_dataset.take(1):
    example = tf.io.parse_single_example(raw_record, feature_desc)
    L = int(example['L'].numpy()[0])
    print(f'[VERIFY] prot_id  : {example["prot_id"].numpy().decode()}')
    print(f'         L        : {L}')
    print(f'         seq_1hot : reshape → ({L}, 26) ✓' if len(example['seq_1hot'].values) == L*26 else 'ERROR')
    print(f'         mf_labels: shape=({n_mf},), sum={example["mf_labels"].numpy().sum()}')
    print(f'         bp_labels: shape=({n_bp},), sum={example["bp_labels"].numpy().sum()}')
    print(f'         cc_labels: shape=({n_cc},), sum={example["cc_labels"].numpy().sum()}')
    print(f'         NO ca_dist_matrix key ✓')

print('\n[OK] TFRecord format verified!')

---
## Phase 8 — Training: Molecular Function (MF)

Train AMPR model cho branch MF (~489 GO terms).  
Config: `configs/mf.yaml`

**Training log format:**
```
[EPOCH 01/50] loss=0.3421 (bce=0.2891 dag=0.0530) | α=[0.612, 0.251, 0.137] | val Fmax=0.412
[EPOCH 02/50] loss=0.3187 (bce=0.2701 dag=0.0486) | α=[0.589, 0.278, 0.133] | val Fmax=0.431
```

> **Thời gian ước tính:** 1–2 giờ (50 epochs, T4 GPU)

In [ ]:
# Cập nhật configs để trỏ đúng data paths (absolute paths trên Colab)
import yaml

def update_config_paths(config_path, work_dir):
    """Update relative paths in config to absolute Colab paths."""
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    for key in cfg['data']:
        if isinstance(cfg['data'][key], str) and cfg['data'][key].startswith('data/'):
            cfg['data'][key] = os.path.join(work_dir, cfg['data'][key])

    for key in cfg['output']:
        if isinstance(cfg['output'][key], str):
            cfg['output'][key] = os.path.join(work_dir, cfg['output'][key])
            os.makedirs(os.path.dirname(cfg['output'][key]), exist_ok=True)

    out_path = config_path.replace('.yaml', '_colab.yaml')
    with open(out_path, 'w') as f:
        yaml.dump(cfg, f)
    return out_path

mf_config = update_config_paths(f'{WORK_DIR}/configs/mf.yaml', WORK_DIR)
bp_config = update_config_paths(f'{WORK_DIR}/configs/bp.yaml', WORK_DIR)
cc_config = update_config_paths(f'{WORK_DIR}/configs/cc.yaml', WORK_DIR)

print(f'Colab configs:')
print(f'  MF: {mf_config}')
print(f'  BP: {bp_config}')
print(f'  CC: {cc_config}')

In [ ]:
# Train MF model
print('=' * 60)
print('TRAINING: Molecular Function (MF)')
print('=' * 60)

!python {WORK_DIR}/main.py --config {mf_config} --seed 42

print('\n[DONE] MF training complete!')
!ls -lh {WORK_DIR}/checkpoints/mf/

---
## Phase 9 — Training: Biological Process (BP)

Train AMPR model cho branch BP (~1943 GO terms).  
BP có nhiều GO terms nhất → training chậm hơn MF.  
DAG loss quan trọng hơn ở BP vì hierarchy sâu hơn.

In [ ]:
print('=' * 60)
print('TRAINING: Biological Process (BP)')
print('=' * 60)

!python {WORK_DIR}/main.py --config {bp_config} --seed 42

print('\n[DONE] BP training complete!')
!ls -lh {WORK_DIR}/checkpoints/bp/

---
## Phase 10 — Training: Cellular Component (CC)

Train AMPR model cho branch CC (~320 GO terms).  
CC có ít GO terms nhất → training nhanh nhất.

In [ ]:
print('=' * 60)
print('TRAINING: Cellular Component (CC)')
print('=' * 60)

!python {WORK_DIR}/main.py --config {cc_config} --seed 42

print('\n[DONE] CC training complete!')
!ls -lh {WORK_DIR}/checkpoints/cc/

---
## Phase 11 — Đánh giá & Visualize Kết quả

Parse training logs để hiển thị:
1. Loss curves (BCE vs DAG loss)
2. Alpha weight evolution qua epochs
3. Validation Fmax curves
4. Bảng kết quả cuối cùng (MF / BP / CC)

In [ ]:
import re
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def parse_log(log_path):
    """Parse training log → dict of metric lists."""
    data = {'epoch': [], 'loss': [], 'bce': [], 'dag': [],
            'alpha_seq': [], 'alpha_3di': [], 'alpha_ppi': [], 'val_fmax': []}

    epoch_re = re.compile(
        r'EPOCH (\d+)/.*loss=([\d.]+).*bce=([\d.]+).*dag=([\d.]+).*'
        r'α=\[([\d.]+), ([\d.]+), ([\d.]+)\].*val Fmax=([\d.]+)'
    )

    if not os.path.exists(log_path):
        print(f'[WARN] Log not found: {log_path}')
        return data

    with open(log_path) as f:
        for line in f:
            m = epoch_re.search(line)
            if m:
                data['epoch'].append(int(m.group(1)))
                data['loss'].append(float(m.group(2)))
                data['bce'].append(float(m.group(3)))
                data['dag'].append(float(m.group(4)))
                data['alpha_seq'].append(float(m.group(5)))
                data['alpha_3di'].append(float(m.group(6)))
                data['alpha_ppi'].append(float(m.group(7)))
                data['val_fmax'].append(float(m.group(8)))
    return data

branches = ['mf', 'bp', 'cc']
logs = {b: parse_log(f'{WORK_DIR}/logs/{b}_train.log') for b in branches}

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.35)

colors = {'mf': '#2196F3', 'bp': '#4CAF50', 'cc': '#FF9800'}

for row, branch in enumerate(branches):
    d = logs[branch]
    if not d['epoch']:
        continue
    ep = d['epoch']

    # Loss curve
    ax1 = fig.add_subplot(gs[row, 0])
    ax1.plot(ep, d['bce'], label='BCE', color=colors[branch])
    ax1.plot(ep, d['dag'], label='DAG', color=colors[branch], linestyle='--', alpha=0.7)
    ax1.set_title(f'{branch.upper()} — Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

    # Alpha weights
    ax2 = fig.add_subplot(gs[row, 1])
    ax2.plot(ep, d['alpha_seq'], label='α_seq', color='#E91E63')
    ax2.plot(ep, d['alpha_3di'], label='α_3di', color='#9C27B0')
    ax2.plot(ep, d['alpha_ppi'], label='α_ppi', color='#FF5722')
    ax2.set_title(f'{branch.upper()} — Alpha Weights')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('α')
    ax2.set_ylim(0, 1); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    # Fmax
    ax3 = fig.add_subplot(gs[row, 2])
    ax3.plot(ep, d['val_fmax'], color=colors[branch], linewidth=2)
    ax3.axhline(max(d['val_fmax']) if d['val_fmax'] else 0,
                color='red', linestyle=':', alpha=0.5, label=f'Best: {max(d["val_fmax"]):.3f}')
    ax3.set_title(f'{branch.upper()} — Val Fmax')
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('Fmax')
    ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

fig.suptitle('AMPR Training Results', fontsize=16, fontweight='bold')
plt.savefig(f'{WORK_DIR}/results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/training_curves.png')

In [ ]:
# Bảng kết quả cuối cùng
print('\n' + '='*65)
print(f'{"Branch":>8} | {"Best Fmax":>10} | {"α_seq":>7} | {"α_3di":>7} | {"α_ppi":>7}')
print('-'*65)

for branch in branches:
    d = logs[branch]
    if not d['epoch']:
        print(f'{branch.upper():>8} | {"(no log)":>10}')
        continue

    best_idx = d['val_fmax'].index(max(d['val_fmax']))
    print(
        f'{branch.upper():>8} | '
        f'{d["val_fmax"][best_idx]:>10.3f} | '
        f'{d["alpha_seq"][best_idx]:>7.3f} | '
        f'{d["alpha_3di"][best_idx]:>7.3f} | '
        f'{d["alpha_ppi"][best_idx]:>7.3f}'
    )

print('='*65)
print('α_* = mean alpha weight at best validation epoch')

In [ ]:
# Final alpha comparison bar chart
import matplotlib.pyplot as plt
import numpy as np

branch_labels = ['MF', 'BP', 'CC']
modalities = ['seq (ProteinBERT)', '3Di (ProstT5)', 'PPI (Node2Vec)']
alphas_final = []

for branch in branches:
    d = logs[branch]
    if d['alpha_seq']:
        alphas_final.append([
            d['alpha_seq'][-1],
            d['alpha_3di'][-1],
            d['alpha_ppi'][-1],
        ])
    else:
        alphas_final.append([0.333, 0.333, 0.334])

x = np.arange(len(branch_labels))
width = 0.25
bar_colors = ['#E91E63', '#9C27B0', '#FF5722']

fig, ax = plt.subplots(figsize=(9, 5))
for i, (mod, color) in enumerate(zip(modalities, bar_colors)):
    vals = [alphas_final[j][i] for j in range(len(branch_labels))]
    bars = ax.bar(x + i*width, vals, width, label=mod, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('GO Branch')
ax.set_ylabel('Alpha Weight')
ax.set_title('AMPR Modality Contribution per GO Branch\n(final epoch mean alpha weights)')
ax.set_xticks(x + width)
ax.set_xticklabels(branch_labels)
ax.set_ylim(0, 0.8)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/results/alpha_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/alpha_comparison.png')

---
## Troubleshooting

| Vấn đề | Giải pháp |
|---|---|
| `ImportError: Bio.Alphabet has been removed` | Chạy cell **Patch DeepFRI script** trước Phase 4 |
| `CUDA out of memory` | Giảm `batch_size` trong config YAML (256 → 128 → 64) |
| `ModuleNotFoundError: ampr` | Chạy lại ô `os.chdir(WORK_DIR)` ở Phase 1 |
| Session Colab reset | Dữ liệu trên Drive vẫn an toàn; chỉ cần mount lại Drive |
| Loss = 0.0 mọi epoch | Kiểm tra `.npy` files có cùng thứ tự protein với `protein_order.json` |
| Alpha weights không thay đổi | Kiểm tra learning rate (`lr: 1e-3`) và DAG loss không bị zero |
| Download FTP timeout | Thử lại hoặc dùng alternative mirror |
| `[PATCH] Không tìm thấy pattern nào` | Script DeepFRI đã được update — không cần patch, tiếp tục bình thường |

---

## Tài liệu tham khảo

- **DeepFRI**: Gligorijević et al. (2021) — structure-based GO prediction
- **ProteinBERT**: Brandes et al. (2022) — protein language model
- **ProstT5**: Heinzinger et al. (2023) — structure-aware protein LM
- **CAFA evaluation**: Zhou et al. (2019) — Fmax, Smin metrics
- **True Path Rule**: GO Consortium — DAG hierarchy constraint
- **Biopython Alphabet migration**: https://biopython.org/wiki/Alphabet

In [ ]:
import shutil

DRIVE_RESULT_DIR = f'/content/drive/MyDrive/ampr_results'
os.makedirs(DRIVE_RESULT_DIR, exist_ok=True)

for branch in branches:
    src_ckpt = f'{WORK_DIR}/checkpoints/{branch}/best.pt'
    if os.path.exists(src_ckpt):
        dst = f'{DRIVE_RESULT_DIR}/{branch}_best.pt'
        shutil.copy2(src_ckpt, dst)
        print(f'[COPY] {src_ckpt} → {dst}')

for fname in ['training_curves.png', 'alpha_comparison.png']:
    src = f'{WORK_DIR}/results/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_RESULT_DIR}/{fname}')
        print(f'[COPY] {src} → Drive')

print('\n[DONE] Tất cả kết quả đã được lưu về Drive!')

---
## Troubleshooting

| Vấn đề | Giải pháp |
|---|---|
| `CUDA out of memory` | Giảm `batch_size` trong config YAML (256 → 128 → 64) |
| `ModuleNotFoundError: ampr` | Chạy lại ô `os.chdir(WORK_DIR)` ở Phase 1 |
| Session Colab reset | Dữ liệu trên Drive vẫn an toàn; chỉ cần mount lại Drive |
| Loss = 0.0 mọi epoch | Kiểm tra `.npy` files có cùng thứ tự protein với `protein_order.json` không |
| Alpha weights không thay đổi | Kiểm tra learning rate (`lr: 1e-3`) và DAG loss không bị zero |
| Download FTP timeout | Thử lại hoặc dùng alternative mirror |

---

## Tài liệu tham khảo

- **DeepFRI**: Gligorijević et al. (2021) — structure-based GO prediction
- **ProteinBERT**: Brandes et al. (2022) — protein language model
- **ProstT5**: Heinzinger et al. (2023) — structure-aware protein LM
- **CAFA evaluation**: Zhou et al. (2019) — Fmax, Smin metrics
- **True Path Rule**: GO Consortium — DAG hierarchy constraint